## Project guide

**Engineering question:** Can the measured oxide thickness meet the specified **560 ± 100 Å** requirement consistently, and what variation patterns deserve investigation?

**Study design:** 21 furnace runs × 4 zones × 2 sampled wafers per run–zone combination = **168 observations**. Each `RUN` is a processing cycle, `ZONE` identifies furnace position, and `THICKNESS` is the *grown oxide film* thickness in Å. The wafer labels do not identify the same physical wafers across runs.

**Scope:** This is an exploratory analysis of thickness measurements, not a causal investigation. Temperature, pressure, process time, gas conditions, and incoming wafer characteristics were not recorded. The notebook follows **EDA → SPC → ANOVA → capability → engineering conclusions**.

**Reading the notebook:** Code cells perform the calculations; the Markdown notes explain why each analysis is useful and how to interpret it. Some early inspection cells are retained to show the original data-loading workflow.

# Furnace Process SPC and Capability Analysis

## Objective

The engineering problem here is that we want to understand if with the given process flow and parameters can an oxide film of 560 Angstroms +/- 100 Angstroms be grown consistently on the silicon wafers. Also to check for sources of variation in the process so that a process control strategy can be created.

## Dataset

Source: NIST/SEMATECH Engineering Statistics Handbook

The data provided represents the oxide thickness measured from different wafers, at differnt zones in the furnace from different batches/ runs.

## 1. Data loading and quality checks

Inspect the source file and load the measurements. The subsequent descriptive statistics help check data structure and measurement ranges. Specification limits are **LSL = 460 Å**, **target = 560 Å**, and **USL = 660 Å**; specification compliance is not the same as statistical control.

In [1]:
import os
os.getcwd()

'C:\\Users\\dhany\\furnace-spc-analysis'

In [2]:
os.listdir("data")

['.ipynb_checkpoints', 'FURNACE.DAT']

In [3]:
with open("data/FURNACE.DAT", "r") as file:
    for i in range(40):
        print(repr(file.readline()))

'This is the Dataplot data file        FURNACE.DAT\n'
'Data provided by Barry Hembre of AMD.  This data set is analyzed\n'
'as a case study in the "Production Process Characterization"\n'
'chapter of the NIST/SEMATECH Engineering Statistics Handbook.\n'
'The URL for this case study is:\n'
'\n'
'   http://www.itl.nist.gov/div898/handbook/ppc/section5/ppc51.htm\n'
'\n'
'Contact--Barry Hembree, AMD\n'
'Number of observations                       = 168\n'
'Number of observations per line image        = 4\n'
'Order of variables on a line image--\n'
'   1. Factor Variable RUN          = Run Number\n'
'   2. Factor Variable ZONE         = Each furnace has 4 "zones"\n'
'   3. Factor Variable WAFER        = Within each zone, 2 wafers are\n'
'                                     measured\n'
'   4. Dependent Variable THICKNESS = Film Thickness (Angstroms)\n'
'  \n'
'To read this file into Dataplot--\n'
'   SKIP 25\n'
'   READ FURNACE.DAT RUN ZONE WAFER THICKNESS\n'
'\n'
'\n'
'RUN   ZONE    WAFER

In [ ]:
import pandas as pd
df = pd.read_csv("data/FURNACE.DAT", sep=r"\s+", skiprows=[*range(23), 24])
df.head()

In [ ]:
df.describe()

## 2. Exploratory data analysis (EDA)

### 2.1 Variation between furnace runs

Each run has eight observations (two wafers in each of four zones). The run-level mean shows changes in average thickness; the run-level standard deviation describes dispersion among those eight observations. Because those eight wafers span four zones, this standard deviation **is not a pure wafer-to-wafer repeatability measure**.

In [ ]:
run_stats = df.groupby("RUN")["THICKNESS"].agg(
    count="count",
    mean="mean",
    std="std",
    min="min",
    max="max"
)

run_stats

In [ ]:
run_stats["deviation_from_target"] = run_stats["mean"] - 560

In [ ]:
run_stats.sort_values("mean", ascending=False)

In [ ]:
import matplotlib.pyplot as plt

run_stats["mean"].plot(
    kind="line",
    marker="o",
    figsize=(10, 4)
)

plt.axhline(560, color="red", linestyle="--", label="Target: 560 Å")
plt.xlabel("Furnace Run")
plt.ylabel("Mean Thickness (Å)")
plt.title("Mean Oxide Thickness by Furnace Run")
plt.legend()
plt.grid(True)
plt.xticks(run_stats.index)

plt.show()

In [ ]:
run_stats["std"].plot(
    kind="bar",
    figsize=(10, 4)
)

plt.xlabel("Furnace Run")
plt.ylabel("Standard Deviation (Å)")
plt.title("Within-Run Thickness Variation")
plt.show()

### 2.2 Differences among zones

Compare thickness distributions by furnace zone. A boxplot shows medians, spread, and possible outliers, but it does not establish whether a difference is statistically significant or what caused it.

In [ ]:
zone_stats = df.groupby("ZONE")["THICKNESS"].agg(
    mean="mean",
    std="std",
    min="min",
    max="max"
)

zone_stats

In [ ]:
import matplotlib.pyplot as plt

df.boxplot(column="THICKNESS", by="ZONE", figsize=(8, 5))

plt.title("Oxide Thickness Distribution by Furnace Zone")
plt.suptitle("")
plt.xlabel("Furnace Zone")
plt.ylabel("Oxide Thickness (Å)")
plt.show()

### 2.3 Run-by-zone patterns

A run–zone mean averages the two wafers sampled in that location during that run. Plotting the four zone series across runs helps identify patterns that may be hidden by an overall mean. The standard deviation of these 21 means per zone measures **run-to-run variation in zone averages**, not the within-pair wafer variation.

In [ ]:
run_zone_stats = df.groupby(["RUN", "ZONE"])["THICKNESS"].agg(
    mean="mean",
    std="std",
    min="min",
    max="max"
)

run_zone_stats.head(12)

In [ ]:
import matplotlib.pyplot as plt

zone_means = df.groupby(["RUN", "ZONE"])["THICKNESS"].mean().unstack()

zone_means.plot(figsize=(11, 5), marker="o")

plt.axhline(560, color="red", linestyle="--", label="Target: 560 Å")
plt.xticks(zone_means.index)
plt.xlabel("Furnace Run")
plt.ylabel("Mean Oxide Thickness (Å)")
plt.title("Oxide Thickness by Furnace Run and Zone")
plt.legend(title="Furnace Zone")
plt.grid(True)
plt.show()

In [ ]:
zone_means.std()

### 2.4 Within-run, within-zone wafer-pair ranges

For each run–zone combination, **range = maximum wafer thickness − minimum wafer thickness**. With two wafers, this is their absolute thickness difference. These ranges describe local within-pair spread; they do not directly quantify all sources of process variation.

In [ ]:
wafer_ranges = df.groupby(["RUN", "ZONE"])["THICKNESS"].agg(
    min="min",
    max="max"
)

wafer_ranges["range"] = wafer_ranges["max"] - wafer_ranges["min"]

wafer_ranges.head(12)

In [ ]:
range_plot = wafer_ranges["range"].unstack()

range_plot.plot(
    kind="line",
    marker="o",
    figsize=(11, 5)
)

plt.xlabel("Furnace Run")
plt.ylabel("Range between two wafers (Å)")
plt.title("Within-Run Wafer Thickness Range by Zone")
plt.xticks(range_plot.index)
plt.grid(True)
plt.legend(title="Furnace Zone")
plt.show()

In [ ]:
wafer_ranges.groupby("ZONE")["range"].agg(
    mean="mean",
    median="median",
    max="max"
)

## 3. Statistical process control (SPC)

### 3.1 R-charts: variability within wafer pairs

Each rational subgroup here is **two wafers sampled in the same zone and run** (`n = 2`). The R-chart tracks their thickness difference across the 21 runs. For `n = 2`, `D3 = 0` and `D4 = 3.267`, so the centre line is the mean range and the control limits are `D3 × R̄` and `D4 × R̄`.

**Interpretation:** Points inside limits are *not* proof that variation is small or absent. They indicate no beyond-limit signal by this rule. Zone 1, Run 20 had a range of **40 Å**, above its approximately **37.96 Å** UCL.

In [ ]:
zone1_ranges = wafer_ranges.xs(1, level="ZONE")["range"]

R_bar = zone1_ranges.mean()

CL = R_bar
UCL = 3.267 * R_bar
LCL = 0

print(f"CL: {CL:.2f} Å")
print(f"UCL: {UCL:.2f} Å")
print(f"LCL: {LCL:.2f} Å")

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(
    zone1_ranges.index,
    zone1_ranges,
    marker="o",
    label="Observed wafer range"
)

plt.axhline(CL, color="green", linestyle="-", label="CL")
plt.axhline(UCL, color="red", linestyle="--", label="UCL")
plt.axhline(LCL, color="red", linestyle="--", label="LCL")

plt.xticks(zone1_ranges.index)
plt.xlabel("Furnace Run")
plt.ylabel("Wafer Thickness Range (Å)")
plt.title("Zone 1 - R Control Chart")
plt.legend()
plt.grid(True)
plt.show()

### 3.2 X̄-charts: changes in subgroup average thickness

Each point is the average of the two wafers in one run–zone subgroup. For subgroup size two, the limits are **grand mean of subgroup means ± 1.880 × R̄**. These are *statistical control limits*, not the engineering specification limits (460–660 Å). An average can be within specifications but outside its control limits.

The first chart develops the calculation for Zone 1; the loop below extends the method to all four zones.

In [ ]:
# Mean thickness of the two wafers in Zone 1, for each run
zone1_means = df[df["ZONE"] == 1].groupby("RUN")["THICKNESS"].mean()

# Centre line
X_double_bar = zone1_means.mean()

# Control limits
A2 = 1.880

X_UCL = X_double_bar + A2 * R_bar
X_LCL = X_double_bar - A2 * R_bar

print(f"CL: {X_double_bar:.2f} Å")
print(f"UCL: {X_UCL:.2f} Å")
print(f"LCL: {X_LCL:.2f} Å")

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(
    zone1_means.index,
    zone1_means,
    marker="o",
    label="Mean thickness"
)

plt.axhline(X_double_bar, color="green", label="CL")
plt.axhline(X_UCL, color="red", linestyle="--", label="UCL")
plt.axhline(X_LCL, color="red", linestyle="--", label="LCL")

plt.axhline(
    560,
    color="purple",
    linestyle=":",
    label="Target: 560 Å"
)

plt.xticks(zone1_means.index)
plt.xlabel("Furnace Run")
plt.ylabel("Mean Oxide Thickness (Å)")
plt.title("Zone 1 - X̄ Control Chart")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
A2 = 1.880
control_limits = {}

for zone in range(1, 5):

    # Calculate mean thickness for each run
    zone_means = df[df["ZONE"] == zone].groupby("RUN")["THICKNESS"].mean()

    # Calculate average wafer-pair range
    zone_ranges = wafer_ranges.xs(zone, level="ZONE")["range"]
    R_bar = zone_ranges.mean()

    # Calculate control limits
    CL = zone_means.mean()
    UCL = CL + A2 * R_bar
    LCL = CL - A2 * R_bar

    # Store results for each zone
    control_limits[zone] = {
        "means": zone_means,
        "R_bar": R_bar,
        "CL": CL,
        "UCL": UCL,
        "LCL": LCL
    }

    print(f"Zone {zone}: CL={CL:.2f}, UCL={UCL:.2f}, LCL={LCL:.2f}")

In [ ]:
import matplotlib.pyplot as plt

for zone in range(1, 5):

    values = control_limits[zone]

    plt.figure(figsize=(10, 4))

    plt.plot(
        values["means"].index,
        values["means"],
        marker="o",
        label="Mean thickness"
    )

    plt.axhline(values["CL"], color="green", label="CL")
    plt.axhline(values["UCL"], color="red", linestyle="--", label="UCL")
    plt.axhline(values["LCL"], color="red", linestyle="--", label="LCL")
    plt.axhline(560, color="purple", linestyle=":", label="Target")

    plt.title(f"Zone {zone} - X̄ Control Chart")
    plt.xlabel("Furnace Run")
    plt.ylabel("Mean Thickness (Å)")
    plt.xticks(range(1, 22))
    plt.legend()
    plt.grid(True)
    plt.show()

### 3.3 R-charts for the remaining zones

The following loop produces the R-charts for Zones 2–4, completing the initial X̄–R chart set. Compare the R-chart findings with the X̄-chart signals: within-pair spread and run-to-run subgroup means answer **different questions**.

In [ ]:
D3 = 0
D4 = 3.267

for zone in range(2, 5):

    # Get the wafer ranges for this zone
    ranges = wafer_ranges.xs(zone, level="ZONE")["range"]

    # Calculate control limits
    R_bar = ranges.mean()
    UCL = D4 * R_bar
    LCL = D3 * R_bar

    # Plot R-chart
    plt.figure(figsize=(10, 4))

    plt.plot(ranges.index, ranges, marker="o",
             label="Observed wafer range")

    plt.axhline(R_bar, color="green", label="CL")
    plt.axhline(UCL, color="red", linestyle="--", label="UCL")
    plt.axhline(LCL, color="red", linestyle="--", label="LCL")

    plt.title(f"Zone {zone} - R Control Chart")
    plt.xlabel("Furnace Run")
    plt.ylabel("Wafer Thickness Range (Å)")
    plt.xticks(range(1, 22))
    plt.legend()
    plt.grid(True)
    plt.show()

    print(f"Zone {zone}: CL={R_bar:.2f}, UCL={UCL:.2f}, LCL={LCL:.2f}")

### 3.4 SPC interpretation and limitations

The X̄-charts exhibit multiple beyond-limit signals, indicating that the process cannot be treated as demonstrated statistically stable on the basis of these data. The R-charts show fewer beyond-limit signals; Zone 1 has a confirmed signal at Run 20, and the other R-charts should be checked numerically before reporting exact counts.

**Important qualifications:** These are **Phase I, provisional limits**, estimated from the same 21 runs under review. The two-wafer subgroup choice, independence, measurement system, distributional assumptions, and chronological sampling details have not been validated. In particular, a subgroup's two wafers may share conditions that make its range much smaller than the variation across runs. Investigate signals before freezing limits; do not remove points without a documented assignable cause.

## 4. Exploratory ANOVA: RUN, ZONE and their interaction

ANOVA tests whether differences among group means are larger than expected relative to residual variation, subject to assumptions. `C(RUN)` and `C(ZONE)` treat their numbers as **categories**; the colon `C(RUN):C(ZONE)` represents an **interaction**: the zone pattern can change from one run to another. `Residual` represents remaining wafer-level differences after fitting these terms.

This is a two-factor **fixed-effects exploratory model**, with two observations per run–zone cell. It does not identify temperature, pressure, wafer batch or other physical causes.

In [ ]:
%pip install statsmodels

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [ ]:
model = ols(
    "THICKNESS ~ C(RUN) + C(ZONE) + C(RUN):C(ZONE)",
    data=df
).fit()

anova_table = sm.stats.anova_lm(model, typ=2)

anova_table

### 4.1 Reading the ANOVA output

- `sum_sq`: sum of squared deviations associated with each term (a measure of variation on the squared-thickness scale).
- `df`: degrees of freedom.
- `F`: mean square for the term divided by the residual mean square.
- `PR(>F)`: p-value, **not** the amount of variation. The conventional **0.05** cutoff is a chosen evidence threshold; subtracting a p-value from 0.05 does not measure variation.

**Observed results:** RUN (`p ≈ 1.89 × 10⁻²⁷`) and RUN × ZONE (`p ≈ 2.95 × 10⁻¹¹`) show strong statistical evidence of differences in this model. The overall ZONE term (`p ≈ 0.0637`) does not cross the conventional 0.05 threshold. Because the interaction is significant, the overall zone effect must be interpreted cautiously.

**Assumptions and scope:** Residual independence, comparable variance and an appropriate error model need checking. Two wafers processed together may be correlated; RUN could also be considered a random effect in a different design. These p-values are **exploratory**, not definitive causal findings.

## 5. Process capability and overall performance

- **Cp** compares specification width with estimated *within-subgroup* process spread; **Cpk** also accounts for where the mean lies relative to the nearest specification limit.
- **Pp** and **Ppk** use the *overall* standard deviation instead of the within-subgroup estimate.
- The conventional formulas use `6σ` for spread, with `3σ` from the mean to the nearest limit for Cpk/Ppk.

Because the control charts indicate instability, these indices are **descriptive/provisional** and must not be interpreted as a validated prediction of future capability. Pooling zones and runs can also conceal subgroup differences.

In [ ]:
# Specification limits
LSL = 460
USL = 660
target = 560

# Overall process mean
process_mean = df["THICKNESS"].mean()

# Within-subgroup variation estimate
# Each subgroup contains 2 wafers
R_bar = wafer_ranges["range"].mean()

# d2 constant for subgroup size n=2
d2 = 1.128
sigma_within = R_bar / d2

print(f"Process mean: {process_mean:.2f} Å")
print(f"Average subgroup range: {R_bar:.2f} Å")
print(f"Estimated within-subgroup SD: {sigma_within:.2f} Å")

In [ ]:
Cp = (USL - LSL) / (6 * sigma_within)

Cpu = (USL - process_mean) / (3 * sigma_within)
Cpl = (process_mean - LSL) / (3 * sigma_within)

Cpk = min(Cpu, Cpl)

print(f"Cp:  {Cp:.3f}")
print(f"Cpk: {Cpk:.3f}")

In [ ]:
sigma_overall = df["THICKNESS"].std(ddof=1)

Pp = (USL - LSL) / (6 * sigma_overall)

Ppk = min(
    (USL - process_mean) / (3 * sigma_overall),
    (process_mean - LSL) / (3 * sigma_overall)
)

print(f"Overall SD: {sigma_overall:.2f} Å")
print(f"Pp:  {Pp:.3f}")
print(f"Ppk: {Ppk:.3f}")

### 5.1 Interpreting the capability estimates

The short-term estimate is derived from the **average of the 84 wafer-pair ranges** divided by `d2 = 1.128` for `n = 2`. It measures *within-pair* dispersion under the chosen subgroup definition; it may underrepresent variation relevant to production across runs.

The overall estimate uses the sample standard deviation of **all 168 thickness measurements**. Compare both sets of indices, but do not claim the furnace is capable until stability and the sampling design have been established.

### Process Capability Analysis — Findings

The furnace oxide thickness data produced Cp = 3.203 and Cpk = 3.106, compared with Pp = 1.313 and Ppk = 1.273.

The large difference between the within-subgroup and overall indices indicates that variation across the full dataset is considerably greater than variation estimated from wafer pairs within the same run and zone.

Although all 168 observed measurements were within the specification limits of 460–660 Å, the SPC analysis identified multiple out-of-control signals in the subgroup means.

Therefore, the calculated capability indices should be treated as exploratory. The data do not establish that the furnace process is statistically stable or capable of consistently meeting specifications in future production.

## 6. Engineering conclusions and recommendations

### Key findings

1. **Observed specification compliance:** All **168 measured oxide thicknesses** were between **460 Å and 660 Å** (observed minimum **487 Å**, maximum **634 Å**). The overall mean was approximately **563.04 Å**, close to the **560 Å** target. This describes the sample only.
2. **Variation structure:** Wafer pairs within the same run and zone were generally more similar than measurements considered across the full dataset. The zone-specific mean pair ranges were approximately **11.62, 12.14, 14.86 and 8.33 Å** for Zones 1–4 respectively. This does not prove that wafer-to-wafer variation is universally small.
3. **SPC signals:** Multiple X̄-chart points were outside provisional limits across zones, while fewer R-chart signals were observed. **Zone 1, Run 20** exceeded the R-chart UCL. These results raise concerns about statistical stability despite observed specification compliance.
4. **Exploratory ANOVA:** RUN and RUN × ZONE were statistically significant under the fitted model (`p < 0.05`); the overall ZONE term was not (`p ≈ 0.0637`). Zone differences may depend on the run, and the statistical assumptions need further evaluation.
5. **Capability versus performance:** **Cp = 3.203, Cpk = 3.106** versus **Pp = 1.313, Ppk = 1.273**. The large gap indicates that total observed dispersion is much greater than the dispersion estimated from within-run–zone wafer pairs. These are not validated future capability claims.

### Engineering interpretation

The dataset suggests that variation operating **beyond the two-wafer subgroup** deserves investigation. Possible contributors include run-level furnace conditions, zone-dependent behaviour, incoming wafer characteristics, or measurement-system effects. **No process parameter measurements or wafer batch identifiers are supplied**, so none of these can be established as a root cause.

### Recommended next steps for an actual production study

1. Verify the measurement system and clarify how the two wafers per run–zone were selected, including their independence and representativeness.
2. Investigate control-chart signals using run histories; record furnace temperature profile, oxidation time, gas composition/flow, pressure, recipe changes, maintenance events, and wafer lot information where available.
3. Reassess subgroup design and consider a variance-components or mixed-effects analysis to distinguish run, zone and within-pair contributions more appropriately.
4. Once special causes are understood and stability is demonstrated with sufficient new data, establish prospective control limits and reassess capability by relevant zone or process stream.

### Limitations

This is a **learning/portfolio analysis** of an observational dataset, not a validated process qualification. We lack detailed sampling design, operating conditions, measurement-system assessment and confirmation of statistical assumptions. The findings support **further investigation**, not a causal diagnosis or a guarantee of future yield.

### Project takeaway

**Meeting specifications in the observed sample is not the same as demonstrating a statistically stable and capable process.** The complementary use of EDA, X̄–R charts, exploratory ANOVA, and Cp/Cpk versus Pp/Ppk helps make that distinction.